# YAMNet Embeddings Extraction

In [ ]:
# Mount Drive & Setup
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import librosa
import tensorflow_hub as hub
from tqdm import tqdm
from google.colab import files

# Adjust these to your actual paths
train_folder = "/content/drive/MyDrive/audio-clustering-2402-mtl-782/Dataset/train_folder"
test_folder  = "/content/drive/MyDrive/audio-clustering-2402-mtl-782/Dataset/test_folder"
train_label_csv = "/content/drive/MyDrive/audio-clustering-2402-mtl-782/Dataset/train_labels.csv"

# Where we'll save .npy
X_yam_train_path = "/content/drive/MyDrive/X_yam_train.npy"
y_train_path     = "/content/drive/MyDrive/y_train.npy"
X_yam_test_path  = "/content/drive/MyDrive/X_yam_test.npy"


Mounted at /content/drive


In [ ]:
# Install & Load YAMNet Model
!pip install tensorflow tensorflow_hub librosa --quiet

yamnet_model = hub.load("https://tfhub.dev/google/yamnet/1")

def extract_yamnet_embedding(wav_path, sr=16000):
    """Load .wav, run YAMNet, return mean-pooled embedding of shape (1024,)."""
    try:
        audio, _ = librosa.load(wav_path, sr=sr)
        if len(audio) == 0:
            return None
        _, embeddings, _ = yamnet_model(audio)
        return np.mean(embeddings.numpy(), axis=0)  # (1024,)
    except Exception as e:
        print(f"⚠️ Error processing {wav_path}: {e}")
        return None


In [ ]:
#  Build YAMNet Embeddings for TRAIN Folder
labels_df = pd.read_csv(train_label_csv)  # columns: [filename, category, ...]
X_train_list = []
y_train_list = []

for _, row in tqdm(labels_df.iterrows(), total=len(labels_df)):
    wav_file = os.path.join(train_folder, row["filename"])
    emb = extract_yamnet_embedding(wav_file)
    if emb is not None:
        X_train_list.append(emb)
        y_train_list.append(row["category"])
    else:
        print(f"⚠️ Skipped embedding for {row['filename']}")

X_yam_train = np.vstack(X_train_list)
y_train = np.array(y_train_list)
print("✅ X_yam_train shape:", X_yam_train.shape)
print("✅ y_train shape:", y_train.shape)

# Save .npy
np.save(X_yam_train_path, X_yam_train)
np.save(y_train_path, y_train)
print(f"✅ Saved: {X_yam_train_path} and {y_train_path}")


100%|██████████| 1500/1500 [08:20<00:00,  3.00it/s]


✅ X_yam_train shape: (1500, 1024)
✅ y_train shape: (1500,)
✅ Saved: /content/drive/MyDrive/X_yam_train.npy and /content/drive/MyDrive/y_train.npy


In [ ]:
#  Build YAMNet Embeddings for TEST Folder
test_files = sorted([f for f in os.listdir(test_folder) if f.endswith('.wav')])
X_test_list = []
skipped_files = []

for fname in tqdm(test_files):
    wav_path = os.path.join(test_folder, fname)
    emb = extract_yamnet_embedding(wav_path)
    if emb is not None:
        X_test_list.append(emb)
    else:
        skipped_files.append(fname)

X_yam_test = np.vstack(X_test_list)
print("✅ X_yam_test shape:", X_yam_test.shape)

# Save .npy
np.save(X_yam_test_path, X_yam_test)
print(f"✅ Saved: {X_yam_test_path}")

if skipped_files:
    print("⚠️ Skipped these test files:", skipped_files)


100%|██████████| 500/500 [01:30<00:00,  5.51it/s]

✅ X_yam_test shape: (500, 1024)
✅ Saved: /content/drive/MyDrive/X_yam_test.npy


In [ ]:
# Download .npy to Local
# If you want them locally on your machine
files.download(X_yam_train_path)
files.download(y_train_path)
files.download(X_yam_test_path)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Audio Spectrogram Transformer Embeddings Extraction

In [ ]:
# Install Libraries
!pip install transformers torchaudio librosa datasets --quiet


In [ ]:
# Import Libraries
import os
import librosa
import torch
import torchaudio
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import adjusted_rand_score
from transformers import ASTFeatureExtractor, ASTModel


In [ ]:
# Load AST Model + Feature Extractor
feature_extractor = ASTFeatureExtractor.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
model = ASTModel.from_pretrained("MIT/ast-finetuned-audioset-10-10-0.4593")
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)


In [ ]:
#  Define AST Embedding Extractor
def extract_ast_embedding(wav_path):
    try:
        waveform, sr = torchaudio.load(wav_path)
        if waveform.shape[1] == 0:
            return None

        resample = torchaudio.transforms.Resample(orig_freq=sr, new_freq=16000)
        waveform = resample(waveform).mean(dim=0).unsqueeze(0)

        inputs = feature_extractor(waveform.squeeze().numpy(), sampling_rate=16000, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model(**inputs)
        # Take the [CLS] token embedding
        return outputs.last_hidden_state[:, 0, :].cpu().numpy().squeeze()
    except Exception as e:
        print(f"⚠️ {wav_path} failed: {e}")
        return None


In [ ]:
#  Load Train Set & Extract AST Embeddings
labels_df = pd.read_csv(label_csv)
X, y = [], []

for _, row in tqdm(labels_df.iterrows(), total=len(labels_df)):
    emb = extract_ast_embedding(os.path.join(train_folder, row['filename']))
    if emb is not None:
        X.append(emb)
        y.append(row['category'])

X = np.vstack(X)
y = np.array(y)
print("✅ Train shape:", X.shape)


In [ ]:
X.shape, y.shape

In [ ]:
np.save('X_train.npy', X)
np.save('y_train.npy', y)


In [ ]:
#  Process Test Set & Predict
test_files = sorted([f for f in os.listdir(test_folder) if f.endswith('.wav')])
X_test, test_ids = [], []

for f in tqdm(test_files):
    emb = extract_ast_embedding(os.path.join(test_folder, f))
    if emb is not None:
        X_test.append(emb)
        test_ids.append(f)

X_test = np.vstack(X_test)
X_test_scaled = scaler.transform(X_test)
X_test_pca = pca.transform(X_test_scaled)


In [ ]:
np.save('X_test.npy', X_test)
from google.colab import files
files.download('X_test.npy')

# Classification and ARI Calculation

Here, you can directly upload the .npy files that we have extracted and provided !

In [ ]:
#  Mount Drive & Set Paths
from google.colab import drive
drive.mount('/content/drive')

# Paths to your fused AST & YAMNet .npy files and label/test folder
X_ast_train_path = "/content/drive/MyDrive/X_train.npy"
X_yam_train_path = "/content/drive/MyDrive/X_yam_train.npy"
y_train_path     = "/content/drive/MyDrive/y_train.npy"
X_ast_test_path  = "/content/drive/MyDrive/X_test.npy"
X_yam_test_path  = "/content/drive/MyDrive/X_yam_test.npy"
test_folder      = "/content/drive/MyDrive/audio-clustering-2402-mtl-782/Dataset/test_folder"


In [ ]:
#  Install & Import Libraries
!pip install scikit-learn xgboost --quiet

import os
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import adjusted_rand_score
import xgboost as xgb

from google.colab import files


In [ ]:
#  Load .npy Data & Fuse (AST + YAMNet)
X_ast_train = np.load(X_ast_train_path)   # shape (n_samples, d_ast)
X_yam_train = np.load(X_yam_train_path)   # shape (n_samples, d_yam)
y_train     = np.load(y_train_path)       # shape (n_samples,)

X_ast_test  = np.load(X_ast_test_path)    # shape (n_test, d_ast)
X_yam_test  = np.load(X_yam_test_path)    # shape (n_test, d_yam)

# Fuse them horizontally
X_train_fused = np.hstack([X_ast_train, X_yam_train])  # shape (n_samples, d_ast + d_yam)
X_test_fused  = np.hstack([X_ast_test,  X_yam_test])   # shape (n_test, d_ast + d_yam)

print("Fused Train shape:", X_train_fused.shape)
print("Fused Test shape: ", X_test_fused.shape)


In [ ]:
#  Label Encode y
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y_train)

print("✅ Encoded labels shape:", y_encoded.shape)
print("Unique classes:", len(np.unique(y_encoded)))


In [ ]:
#  Standardize the fused embeddings
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_fused)
X_test_scaled  = scaler.transform(X_test_fused)


In [ ]:
#  Tune PCA across multiple dimensions
best_ari = 0
best_pca_model = None
best_X_pca = None
best_n_components = 0

for n in [60, 80, 100, 110, 120, 130, 140, 150, 160, 170, 180, 190, 200, 210, 220]:
    pca = PCA(n_components=n, random_state=42)
    X_pca_candidate = pca.fit_transform(X_train_scaled)

    # Quick baseline check on each dimension with simple LR
    X_train_p, X_val_p, y_train_p, y_val_p = train_test_split(
        X_pca_candidate, y_encoded,
        test_size=0.2, stratify=y_encoded, random_state=42
    )

    # Quick logistic baseline
    clf = LogisticRegression(max_iter=2000, solver='saga')
    clf.fit(X_train_p, y_train_p)
    ari_candidate = adjusted_rand_score(y_val_p, clf.predict(X_val_p))

    print(f"🔍 PCA {n}D → ARI = {ari_candidate:.4f}")

    if ari_candidate > best_ari:
        best_ari = ari_candidate
        best_X_pca = X_pca_candidate
        best_pca_model = pca
        best_n_components = n

print(f"\n✅ Best PCA Dim = {best_n_components}, baseline ARI = {best_ari:.4f}")


In [ ]:
#  Train/Val split with best PCA
X_pca_trainval = best_X_pca
X_train, X_val, y_train_, y_val_ = train_test_split(
    X_pca_trainval, y_encoded,
    test_size=0.2, stratify=y_encoded, random_state=42
)


In [ ]:
#  Tune base classifiers (RF, LR, XGBoost)

# --- A. Random Forest ---
rf_grid = {
    'n_estimators': [100, 200],
    'max_depth': [20, None],
    'min_samples_split': [2, 5]
}
rf_search = GridSearchCV(RandomForestClassifier(random_state=42),
                         rf_grid,
                         cv=3, scoring='accuracy', n_jobs=-1)
rf_search.fit(X_train, y_train_)
clf_rf = rf_search.best_estimator_
print("✅ RF Best Params:", rf_search.best_params_)

# --- B. Logistic Regression ---
lr_grid = {
    'C': [0.1, 1, 10],
    'solver': ['lbfgs', 'saga'],
    'penalty': ['l2'],
    'max_iter': [5000]
}
lr_search = GridSearchCV(LogisticRegression(),
                         lr_grid,
                         cv=3, scoring='accuracy', n_jobs=-1)
lr_search.fit(X_train, y_train_)
clf_lr = lr_search.best_estimator_
print("✅ LR Best Params:", lr_search.best_params_)

# --- C. XGBoost ---
xgb_grid = {
    'n_estimators': [100, 200],
    'max_depth': [6, 10],
    'learning_rate': [0.05, 0.1]
}
xgb_base = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(np.unique(y_encoded)),
    eval_metric='mlogloss',
    use_label_encoder=False,
    random_state=42
)

xgb_search = GridSearchCV(xgb_base,
                          xgb_grid,
                          cv=3, scoring='accuracy', n_jobs=-1)
xgb_search.fit(X_train, y_train_)
clf_xgb = xgb_search.best_estimator_
print("✅ XGB Best Params:", xgb_search.best_params_)


In [ ]:
#  Ensemble (Soft Voting)
# We can start with equal weights or do a manual weighting if we see one model is better
ensemble = VotingClassifier(
    estimators=[
        ('rf', clf_rf),
        ('lr', clf_lr),
        ('xgb', clf_xgb)
    ],
    voting='soft'  # probabilities
    # weights=[1, 2, 2], # optional if you want weighting
)
ensemble.fit(X_train, y_train_)

# Validation ARI
y_val_pred = ensemble.predict(X_val)
ari_ensemble = adjusted_rand_score(y_val_, y_val_pred)
print(f"✅ Ensemble Validation ARI: {ari_ensemble:.4f}")


In [ ]:
# Predict on Test Set
X_test_pca = best_pca_model.transform(X_test_scaled)
y_test_pred = ensemble.predict(X_test_pca)
y_test_labels = label_encoder.inverse_transform(y_test_pred)


In [ ]:
#  Save & Download Submission
test_files = sorted([f for f in os.listdir(test_folder) if f.endswith('.wav')])
submission_df = pd.DataFrame({'id': test_files, 'category': y_test_labels})
submission_df.to_csv("submission_concatenate.csv", index=False)
print("✅ submission_concatenate.csv saved!")

files.download("submission_concatenate.csv")


# Alternate Ideas

Here are some alternate ideas that did not perform as good as the above

Ensembling simply on AST embeddings, rest remains the same ; this gave 0.9532

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# X_train_path = "/content/drive/MyDrive/X_train.npy"
# y_train_path = "/content/drive/MyDrive/y_train.npy"
# X_test_path  = "/content/drive/MyDrive/X_test.npy"
# test_folder  = "/content/drive/MyDrive/audio-clustering-2402-mtl-782/Dataset/test_folder"


# !pip install scikit-learn xgboost --quiet


# X = np.load(X_train_path)
# y = np.load(y_train_path)
# X_test = np.load(X_test_path)

# label_encoder = LabelEncoder()
# y_encoded = label_encoder.fit_transform(y)

# print("✅ Data shapes — X:", X.shape, "| y:", y_encoded.shape, "| X_test:", X_test.shape)


# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)
# X_test_scaled = scaler.transform(X_test)

# best_pca_ari = 0
# best_n_components = 120
# best_X_pca = None
# best_pca_model = None

# for n in [100,102, 105, 107, 110, 112, 115, 117, 120, 122, 125]:
#     pca = PCA(n_components=n, random_state=42)
#     X_pca = pca.fit_transform(X_scaled)
#     X_train, X_val, y_train, y_val = train_test_split(X_pca, y_encoded, test_size=0.2, stratify=y_encoded, random_state=42)

#     clf = LogisticRegression(max_iter=3000, solver='saga')
#     clf.fit(X_train, y_train)
#     ari = adjusted_rand_score(y_val, clf.predict(X_val))

#     print(f"🔍 PCA {n}D → ARI = {ari:.4f}")

#     if ari > best_pca_ari:
#         best_pca_ari = ari
#         best_n_components = n
#         best_X_pca = X_pca
#         best_pca_model = pca

# print(f"\n✅ Best PCA Dim = {best_n_components} with ARI = {best_pca_ari:.4f}")


Used GridSearchCV to determine the optimal weights of ensemble learning classifiers, however this gave an ARI score of 0.9444 ;  rest remains the same

In [ ]:
# from itertools import product

# best_weighted_ari = 0
# best_weights = (1, 1, 1)
# best_ensemble = None

# for w_rf, w_lr, w_xgb in product(range(1, 6), repeat=3):
#     ensemble = VotingClassifier(
#         estimators=[('rf', clf_rf), ('lr', clf_lr), ('xgb', clf_xgb)],
#         voting='soft',
#         weights=[w_rf, w_lr, w_xgb]
#     )
#     ensemble.fit(X_train, y_train)
#     y_val_pred = ensemble.predict(X_val)
#     ari = adjusted_rand_score(y_val, y_val_pred)

#     if ari > best_weighted_ari:
#         best_weighted_ari = ari
#         best_weights = (w_rf, w_lr, w_xgb)
#         best_ensemble = ensemble

# print(f"\n✅ Best Weights: {best_weights}, Validation ARI = {best_weighted_ari:.4f}")
